### Conclusión

Por una parte lo bueno, ha sido rápido. Pero la realidad es que este método no ha sido capaz de superar a nada de lo ya probado y de hecho ha sido un poquito pero. 

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 777
np.random.seed(SEED)

In [11]:

# cargo el dataset básico primero para separarlo en train y test
basic = pd.read_csv('../data/scaled/data_basic.csv')


# separación de características y objetivo
X_basic = basic.drop('Variable de Salida', axis=1)
y_basic = basic['Variable de Salida']

# separación en train y test
X_train_basic, X_test_basic, y_train, y_test = train_test_split(X_basic, y_basic, test_size=0.2, random_state = SEED, shuffle=True, stratify=y_basic)


In [12]:

# Entrenar IsolationForest solo con NOK (clase 1)
X_train_nok = X_train_basic[y_train == 1]


iso_predictor = IsolationForest(
    n_estimators=300,
    contamination='auto',
    random_state=SEED
)

iso_predictor.fit(X_train_nok)

# Score continuo sobre test completo
scores = iso_predictor.score_samples(X_test_basic)

# Búsqueda de umbral óptimo sobre F1 macro
thresholds = np.percentile(scores, np.arange(5, 70, 0.5))
best_t, best_f1 = None, 0

for t in thresholds:
    raw = (scores < t).astype(int)
    preds = 1 - raw  # anomalía=0 (OK), normal=1 (NOK)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

# Predicción final con mejor umbral
raw_final = (scores < best_t).astype(int)
y_pred_iso = 1 - raw_final

print(f"Umbral óptimo: {best_t:.4f}")
print(f"F1 macro:      {best_f1:.4f}\n")
print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso))

Umbral óptimo: -0.4610
F1 macro:      0.4981

[[ 111  377]
 [ 349 1162]]
              precision    recall  f1-score   support

           0       0.24      0.23      0.23       488
           1       0.76      0.77      0.76      1511

    accuracy                           0.64      1999
   macro avg       0.50      0.50      0.50      1999
weighted avg       0.63      0.64      0.63      1999

